# Differential Equations — Session 22
## Section 4.10: Nonlinear Higher-Order Equations

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. Contrast linear and nonlinear solution structure.
2. reduce order when $y$ is missing.
3. reduce order when $x$ is missing using $y''=u\,du/dy$.
4. construct local Taylor approximations.
5. compare Taylor and numerical solutions.
6. analyze an autonomous nonlinear oscillator using energy and phase portraits.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–15 min | Linear versus nonlinear structure |\n| 15–38 min | Missing-variable reductions |\n| 38–58 min | Taylor approximation |\n| 58–78 min | Numerical comparison |\n| 78–88 min | Autonomous oscillator and energy |\n| 88–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

### Nonlinearity

Superposition generally fails for nonlinear equations. Closed-form general solutions are rare, so qualitative and numerical methods are central.

### Reduction when $y$ is missing

For $F(x,y',y'')=0$, set

$$u(x)=y'(x),\qquad y''=u'.$$

Solve the first-order equation for $u$, then integrate $y'=u$.

### Reduction when $x$ is missing

For $F(y,y',y'')=0$, set $u(y)=y'$. By the chain rule,

$$y''=\frac{du}{dx}=\frac{du}{dy}\frac{dy}{dx}=u\frac{du}{dy}.$$

### Taylor approximation

Repeatedly differentiate the equation and use initial data to determine derivatives at the expansion point.

### Energy principle

For an autonomous conservative equation

$$y''+V'(y)=0,$$

$$E=\frac12(y')^2+V(y)$$

is constant along solutions.

### Classroom Checkpoint — Reduce Order in a Nonlinear Equation

If a second-order equation contains $x$, $y'$, and $y''$ but not $y$, which substitution is natural?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Superposition failure

Both $y_1$ and $y_2$ may solve a nonlinear equation while $y_1+y_2$ does not.

In [ ]:
x=sp.symbols('x'); y1=1/(1-x); y2=1/(2-x)
def N(y): return sp.simplify(sp.diff(y,x)-y**2)
display(N(y1)); display(N(y2)); display(sp.simplify(N(y1+y2)))

## 2. Dependent variable missing

Solve

$$y''=(y')^2.$$

Set $u=y'$. Then $u'=u^2$, so

$$u=-\frac{1}{x+C_1},\qquad y=-\ln|x+C_1|+C_2.$$

## 3. Independent variable missing

For

$$yy''=(y')^2,$$

set $u(y)=y'$. Then

$$yu\frac{du}{dy}=u^2.$$

For $u\ne0$, $du/u=dy/y$, so $u=Cy$ and $y=Ae^{Cx}$. Constant solutions must be checked separately.

In [ ]:
x=sp.symbols('x'); A,C=sp.symbols('A C'); y=A*sp.exp(C*x); display(sp.simplify(y*sp.diff(y,x,2)-sp.diff(y,x)**2))

## 4. Taylor approximation for a nonlinear IVP

Consider

$$y''=-y-y^3,\qquad y(0)=1,\quad y'(0)=0.$$

A power-series recurrence can be obtained by substituting

$$y=\sum_{n=0}^Na_nx^n.$$

In [ ]:
x=sp.symbols('x'); N=10; coeff=sp.symbols('a0:'+str(N+1)); series=sum(coeff[n]*x**n for n in range(N+1)); expr=sp.series(sp.diff(series,x,2)+series+series**3,x,0,N-1).removeO(); equations=[sp.Eq(coeff[0],1),sp.Eq(coeff[1],0)]+[sp.Eq(sp.expand(expr).coeff(x,k),0) for k in range(N-1)]; sol=sp.solve(equations,coeff,dict=True)[0]; poly=sp.expand(series.subs(sol)); display(poly)

In [ ]:
# Compare Taylor polynomial with numerical solution
poly_fun=sp.lambdify(x,poly,'numpy'); soln=solve_second_order(lambda t,y,v:-y-y**3,(-3,3),1,0,points=1000,rtol=1e-10,atol=1e-12)
plt.plot(soln.t,soln.y[0],label='numerical'); plt.plot(soln.t,poly_fun(soln.t),linestyle='--',label='Taylor polynomial'); plt.ylim(-3,3); plt.legend(); plt.show()

## 5. Nonlinear oscillator and amplitude-dependent period

For

$$y''+y+y^3=0,$$

$$E=\frac12(y')^2+\frac12y^2+\frac14y^4$$

is conserved. Unlike the linear oscillator, the period depends on amplitude.

In [ ]:
def nonlinear_oscillator(A=1.0,T=30):
    sol=solve_second_order(lambda t,y,v:-y-y**3,(0,T),A,0,points=1800,rtol=1e-9,atol=1e-11)
    y,v=sol.y; E=.5*v**2+.5*y**2+.25*y**4
    plt.plot(sol.t,y); plt.title('nonlinear oscillation'); plt.show(); plt.plot(y,v); plt.xlabel('y'); plt.ylabel("y'"); plt.title('energy level in phase plane'); plt.show(); print('max energy drift',np.max(np.abs(E-E[0])))
if WIDGETS_AVAILABLE: interact(nonlinear_oscillator,A=FloatSlider(min=.1,max=3,step=.1,value=1),T=IntSlider(min=10,max=60,step=5,value=30))
else: nonlinear_oscillator()

## Classroom Checkpoint — Exit Check

Which substitution is appropriate for

$$y''+y(y')^2=0?$$

> Pause here. Let students commit to an answer before running the next cell.